# Exercises XP – Week 7, Day 2: Deep Dive into Advanced Transformer and LLM Concepts
Written exercises (no code needed).

## 🌟 Exercise 1: Traditional vs. Modern NLP

| Aspect | Traditional NLP | Modern NLP |
|---|---|---|
| **Feature Engineering** | **Manual**: experts design features (bag-of-words, TF-IDF, n-grams, part-of-speech tags, hand-written rules). | **Automatic**: neural networks learn features directly from raw text. |
| **Word Representations** | **Static** or none: each word has one fixed vector (one-hot, TF-IDF, or Word2Vec/GloVe), whatever the context. "Bank" is the same in "river bank" and "bank account". | **Contextual**: a word's vector depends on the whole sentence (BERT, GPT), so "bank" gets different representations in different sentences. |
| **Model Architectures** | **Shallow**: Naïve Bayes, logistic regression, SVMs, HMMs/CRFs; later small RNNs. | **Deep**: Transformers with many layers and self-attention, from millions to hundreds of billions of parameters. |
| **Training Methodology** | **Task-specific**: a new model is trained from scratch for each task, on that task's labelled data. | **Pre-training + fine-tuning**: one model is pre-trained on huge unlabelled text, then adapted to many tasks with little labelled data, or just with prompts. |
| **Key Examples** | Naïve Bayes spam filters, TF-IDF + SVM classifiers, HMM part-of-speech taggers, CRF named-entity recognisers, Word2Vec. | BERT, RoBERTa, GPT-2/3/4, T5, LLaMA, Claude. |
| **Advantages** | Fast and cheap, works with small data, easy to interpret, runs on a CPU. | Much higher accuracy, understands context and nuance, one model serves many tasks, works across languages. |
| **Disadvantages** | Misses context and word order, needs feature engineering for each task, struggles with ambiguity and rare words. | Expensive to train and run (GPUs), needs huge data, hard to interpret, can inherit biases from training data and produce confident errors ("hallucinations"). |

**Impact on scalability and efficiency:** in traditional NLP, every new task, language or domain meant collecting new labelled data, designing new features and training a new model, so it scaled poorly. Modern NLP turned this around: the expensive part (pre-training) is done **once**, and the same model can then be fine-tuned for a new task with a few thousand examples, or even used **zero-shot** with a prompt. That makes building new NLP applications far faster and cheaper for most teams, and Transformers' parallel processing lets models scale to enormous datasets. The price is that running these models is computationally heavy, so efficiency techniques (distillation, quantisation, parameter-efficient fine-tuning like LoRA) have become essential.

## 🌟 Exercise 2: LLM Architecture and Application Scenarios

**BERT (Bidirectional Encoder Representations from Transformers)**
- *Architecture:* **encoder-only**, **bidirectional**: every word attends to the words both before and after it. Pre-trained with **Masked Language Modelling** (predict randomly hidden words).
- *Where it excels:* **understanding tasks**, e.g. **classifying customer support tickets** by topic and urgency, sentiment analysis, named-entity recognition, or search relevance.
- *Why:* to classify or label text you need to understand the **whole input at once**, with full context on both sides. BERT builds a rich representation of every word using all of its context, which is exactly what classification needs. It doesn't need to generate text.

**GPT (Generative Pre-trained Transformer)**
- *Architecture:* **decoder-only**, **unidirectional / causal**: each token can only see the tokens before it. Pre-trained with **Causal Language Modelling** (predict the next token).
- *Where it excels:* **open-ended text generation**, e.g. a **conversational assistant** or drafting emails and creative writing.
- *Why:* generation happens one token at a time from left to right, which is exactly the task GPT was trained on. Its causal design produces fluent, coherent continuations of any prompt.

**T5 (Text-to-Text Transfer Transformer)**
- *Architecture:* **encoder-decoder**: a bidirectional encoder reads the full input, and a causal decoder generates the output. Every task is framed as **text-to-text** ("translate English to German: …", "summarize: …"). Pre-trained by filling in masked spans of text.
- *Where it excels:* **transforming one text into another**, e.g. **summarising long news articles** or translation.
- *Why:* these tasks need **both** a full understanding of the input (encoder) **and** fluent generation of a new text that can be very different in length and wording (decoder). The encoder-decoder split is built for this.

## 🌟 Exercise 3: The Benefits and Ethical Considerations of Pre-training

**Five key benefits of pre-trained models:**
1. **Improved generalisation:** after seeing a huge variety of text, the model learns general patterns of language that carry over to new topics and phrasings it never saw in a specific task's data.
2. **Reduced need for labelled data:** the model already understands language, so a task like sentiment analysis needs only hundreds or thousands of labelled examples instead of millions. Labelling is expensive, so this matters a lot.
3. **Faster fine-tuning:** starting from good weights, the model reaches high performance after a few epochs (minutes to hours) instead of being trained from scratch (days to weeks).
4. **Transfer learning:** knowledge learned in one setting is reused in another: one pre-trained model can become a classifier, a question-answerer, or a summariser. Multilingual models even transfer skills learned in English to other languages.
5. **Robustness:** exposure to diverse, messy text (typos, slang, different styles) makes the model more tolerant of noisy real-world input than a model trained on a small, clean dataset.

**Ethical concerns:**
- **Bias:** internet text contains stereotypes about gender, ethnicity, religion and more. The model learns and can amplify them (e.g. associating certain jobs with one gender), which can lead to unfair outcomes in hiring or lending tools.
- **Misinformation:** models learn false claims along with true ones, and they can produce fluent, confident but wrong answers ("hallucinations").
- **Misuse:** text generation can be used for spam, phishing, propaganda, fake reviews or impersonation at scale.
- **Privacy and copyright:** training data may contain personal information or copyrighted text that the model can reproduce.
- **Environmental cost:** pre-training uses huge amounts of energy.

**Mitigation strategies:**
- **Curate and document training data:** filter toxic or low-quality content, remove personal data, and publish "datasheets" describing what's in the data.
- **Measure and reduce bias:** test with bias benchmarks, balance training data, and use debiasing techniques during fine-tuning.
- **Alignment and safety training:** fine-tune with human feedback (RLHF) or principle-based methods so models refuse harmful requests and admit uncertainty.
- **Grounding and verification:** connect models to trusted sources (retrieval-augmented generation) and show citations to reduce hallucinations.
- **Usage policies and monitoring:** access controls, abuse detection, and watermarking or labelling AI-generated content.
- **Human oversight in high-stakes domains:** in medicine, law, finance or hiring, keep a human reviewing important decisions.

## 🌟 Exercise 4: Transformer Architecture Deep Dive

### Self-Attention and Multi-Head Attention
**How self-attention works:** each token's embedding is multiplied by three learned matrices to create a **query (Q)**, a **key (K)** and a **value (V)** vector. For a given token, we compare its query with the keys of **all** tokens (dot product), which gives a relevance score for every other token. The scores are divided by √dₖ and passed through a **softmax**, so they become weights that add up to 1. The token's new representation is the **weighted sum of all the value vectors**. In short, every token gathers information from the tokens most relevant to it. This happens for all tokens in parallel: **Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V**.

**Why multi-head attention:** a single attention head can only produce one weighting pattern per token, so it has to compress all kinds of relationships into one average. **Multi-head attention** runs several attention operations in parallel (e.g. 12 heads in BERT-base), each with its own Q/K/V matrices in a smaller sub-space, then concatenates and mixes their outputs. Each head can specialise in a different type of relationship (grammar, meaning, position, co-reference), which gives a richer representation for about the same computational cost.

**Example:** *"The nurse told the patient that she would bring his medication after lunch."*
- **Head 1 – co-reference:** "she" attends strongly to "nurse", and "his" attends to "patient", working out who is who.
- **Head 2 – verb–object relations:** "bring" attends to "medication" (what is brought) and "told" attends to "patient" (who was told).
- **Head 3 – time and order:** "after" attends to "lunch", and "would" attends to "bring", capturing future timing.
- **Head 4 – local syntax:** each word attends mostly to its neighbours ("the" → "nurse", "his" → "medication"), capturing phrase structure.

Combining the heads, the representation of "she" now contains "it's the nurse, who is doing the bringing, later".

### Pre-training Objectives
- **Masked Language Modelling (MLM)** (BERT): ~15% of tokens are hidden, and the model predicts them using context from **both sides**. This gives deep bidirectional understanding, but the model isn't naturally a text generator.
- **Causal Language Modelling (CLM)** (GPT): the model predicts the **next token** from the previous ones only. Its context is one-directional, but it can generate text naturally, token by token.

*MLM is more appropriate* for **understanding tasks**, e.g. extracting medication names and dosages from clinical notes, where each word should be interpreted using the full sentence.
*CLM is more appropriate* for **generation**, e.g. an assistant that drafts replies to patient questions or writes product descriptions.

**Next Sentence Prediction (NSP):** early BERT was also trained to predict whether sentence B really follows sentence A. The idea was to help with tasks involving sentence pairs (question answering, natural language inference). Later studies, notably **RoBERTa**, showed NSP brought little or no benefit and could even hurt: the task was too easy (random sentences are usually about a different topic, so the model learned topic matching rather than coherence), and training on single-sentence pairs limited the use of long contexts. Modern models drop NSP and use longer contiguous text instead, or harder alternatives such as ALBERT's sentence-order prediction.

### Transformer Model Selection
- **Customer review sentiment (positive / negative / neutral) → encoder-only (e.g. BERT, RoBERTa, DistilBERT).** The task is understanding and classifying a full input text, with no generation needed. Bidirectional context captures negations and nuance ("not bad at all"), and encoders are smaller and faster for classification.
- **Creative conversational chatbot → decoder-only (e.g. GPT, LLaMA).** The task is open-ended generation, continuing a conversation token by token. Decoder-only models are trained exactly for this and scale well to large, fluent, creative models.
- **Translating technical documents from English to Spanish → encoder-decoder (e.g. T5, mBART, MarianMT).** The encoder fully understands the English sentence (including technical terms and structure), and the decoder generates the Spanish version, which may have a different length and word order. This is the classic sequence-to-sequence setup for translation.

### Positional Encoding
**Purpose:** self-attention treats its input as a **set**: it compares every token with every other token but has no built-in sense of order. Without extra information, "dog bites man" and "man bites dog" would produce the same representations. **Positional encodings** (fixed sine/cosine patterns, or learned position embeddings) are added to the token embeddings so the model knows where each token is and can learn order-dependent meaning, grammar and distance between words.

**Example of a problem without it:** in *"The doctor didn't say the test was positive"* vs. *"The doctor said the test wasn't positive"*, or *"Alice paid Bob"* vs. *"Bob paid Alice"*, the words are nearly or exactly the same and only the order differs. A model without positional information couldn't tell who paid whom, which could flip the meaning of a transaction record or a medical note.

## 🌟 Exercise 5: BERT Variations – Choose Your Detective

| Scenario | Best choice | Why |
|---|---|---|
| **1. Real-time sentiment analysis on a mobile app with limited resources** | **DistilBERT** | About 40% smaller and 60% faster than BERT while keeping ~97% of its performance, which makes real-time inference on a phone practical. |
| **2. Research on legal documents requiring high accuracy** | **RoBERTa** | A more robustly trained BERT (more data, longer training, dynamic masking, no NSP) that gives top accuracy on understanding tasks. Accuracy matters more than speed here. |
| **3. Global customer support in multiple languages** | **XLM-RoBERTa** | Pre-trained on text in 100 languages, so one model understands and classifies messages across languages, with strong cross-lingual transfer. |
| **4. Efficient pre-training and token-replacement detection** | **ELECTRA** | Trained as a *discriminator* that detects which tokens were replaced by a small generator. It learns from **every** token rather than only the 15% masked ones, so pre-training is much more efficient. |
| **5. Efficient NLP in resource-constrained environments** | **ALBERT** | Shares parameters across layers and factorises the embedding matrix, so the model has far fewer parameters and a small memory footprint (though it isn't faster to run than BERT). *(DistilBERT is the alternative when speed is the main constraint.)* |

**Comparison of BERT variations:**

| Model | Training data & method | Size & efficiency | Key optimisations / innovations | Ideal use cases |
|---|---|---|---|---|
| **BERT** (baseline) | BooksCorpus + English Wikipedia (~16 GB); MLM + NSP | Base 110M, Large 340M parameters | Bidirectional Transformer encoder pre-training | General English understanding tasks |
| **RoBERTa** | ~160 GB of text; MLM only, longer training, bigger batches | Same size as BERT (125M / 355M) | Removes NSP, dynamic masking, better hyperparameters | Highest accuracy when compute is available (legal, research, benchmarks) |
| **ALBERT** | Same data as BERT; MLM + Sentence-Order Prediction | Very few parameters (base ~12M); small memory, similar compute | Cross-layer parameter sharing, factorised embeddings, SOP instead of NSP | Memory-limited settings; scaling up models cheaply |
| **DistilBERT** | Distilled from BERT on the same data (student learns from teacher) | 66M parameters; ~40% smaller, ~60% faster, ~97% of BERT's performance | Knowledge distillation, 6 layers instead of 12 | Mobile, real-time, low-latency or low-cost inference |
| **ELECTRA** | Same kind of data; replaced-token detection (generator + discriminator) | Small/Base/Large; strong results with much less pre-training compute | Learns from all tokens, very sample-efficient pre-training | Efficient pre-training, strong accuracy at small model sizes |
| **XLM-RoBERTa** | 2.5 TB of CommonCrawl text in 100 languages; MLM | Base 270M, Large 550M (big multilingual vocabulary) | Massive multilingual training, RoBERTa-style setup | Multilingual and cross-lingual tasks (global support, translation-related classification) |

## 🌟 Exercise 6: Softmax Temperature – The Randomness Regulator

### 1. Temperature scenarios
The model's raw scores (logits) are divided by the temperature before the softmax: **p = softmax(logits / T)**.
- **T = 0.2 (low):** the distribution becomes very **sharp**, so the most likely token gets almost all the probability. Output is **focused, predictable and repetitive**, close to always picking the top word. It's good for facts, but it can sound dull or loop.
- **T = 1 (default):** the model's original probabilities are used unchanged, a **balanced** mix of likely and less likely words. Output is natural but varied.
- **T = 1.5 (high):** the distribution becomes **flatter**, so unlikely tokens get much more chance. Output is **more creative and surprising, but also less coherent**: it can drift off-topic, make up facts, or produce odd word choices.

### 2. Application design
- **Personalised bedtime stories for children:** use a **moderately high temperature (around 0.8–1.0)** so stories feel fresh and imaginative, with different characters and twists each night, but not so high that the story stops making sense. Combine it with **top-p (nucleus) sampling** to cut off very unlikely words, plus a content filter for child-safe language. You could even raise the temperature slightly for "adventure" parts and lower it for the calm ending.
- **Summaries of financial reports:** use a **low temperature (0–0.3)** or near-greedy decoding, so the model sticks to the most likely, faithful wording and doesn't invent numbers. Add grounding: instruct the model to use only figures from the source, check every number in the summary against the report automatically, and have a human review before publishing. Here reliability matters far more than creativity.

### 3. Temperature and bias
- **Low temperature** makes the model pick its **most probable** continuation every time. If the training data contained a stereotype, the most probable completion may be the stereotyped one, and low temperature makes it appear **consistently**, which amplifies the bias.
- **High temperature** spreads probability more evenly, so outputs are more varied and the single dominant stereotype appears less often. But it also gives more chance to **rare toxic or offensive** completions that low temperature would almost never pick.
- **Practical example:** for the prompt *"The nurse walked in and"*, a model at **T = 0.2** might continue with *"she"* almost 100% of the time, reinforcing the stereotype that nurses are women. At **T = 1.2**, *"he"* or *"they"* appear more often, but you might also occasionally get inappropriate or nonsensical continuations. Temperature changes **how often** biased outputs appear, but it doesn't **remove** the bias learned by the model. That needs better data, fine-tuning and evaluation.